<a href="https://colab.research.google.com/github/Aerospace87/ML-projects/blob/main/genai/RAGExample.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

## RAG Pipeline

1. The user inputs a question
2. The pipeline retrieves the most similar documents to the question
3. The pipeline passes the question and the retrieved documents to the
LLM
4. The pipeline generates a response

## Dependencies

In [ ]:
pip install -U sentence-transformers;

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 345.7/345.7 kB 6.2 MB/s eta 0:00:00:00:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 363.4/363.4 MB 4.6 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 664.8/664.8 MB 2.5 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 211.5/211.5 MB 7.9 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 56.3/56.3 MB 30.7 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 127.9/127.9 MB 12.9 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 207.5/207.5 MB 6.1 MB/s eta 0:00:000:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 21.1/21.1 MB 70.2 MB/s eta 0:00:00:00:0100:01
  Attempting uninstall: nvidia-nvjitlink-cu12
    Found existing installation: nvidia-nvjitlink-cu12 12.9.41
    Uninstalling nvidia-nvjitlink-cu12-12.9.41:
      Successfully uninstalled nvidia-nvjitlink-cu12-12.9.41
  Attempting uninstall: nvidia-curand-cu12
    Found exi

In [ ]:
## This is only to use the notebook in kaggle. In Google Colab is not necessary. You need to create secret first
import os
from kaggle_secrets import UserSecretsClient
user_secrets = UserSecretsClient()

# Set your HF token & username as environment variables
os.environ["HF_TOKEN"] = user_secrets.get_secret("HF_TOKEN")
# Replace with your username)
os.environ["HF_USERNAME"] = "Gianloco"

In [ ]:
import numpy as np
import torch
from sentence_transformers import SentenceTransformer
from transformers import AutoModelForCausalLM, AutoTokenizer, pipeline

2025-06-18 14:23:15.378244: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:477] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1750256595.817362      35 cuda_dnn.cc:8310] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1750256595.924054      35 cuda_blas.cc:1418] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered


## Functions

In [ ]:
# 1. Load a pretrained Sentence Transformer model
sentence_transformer = SentenceTransformer("all-MiniLM-L6-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [ ]:
def embed_documents(documents: list[str]):
    # Use a sentence transformer model to encode the documents
    # Store the documents somewhere

    # 2. Calculate embeddings by calling model.encode()
    embeddings = sentence_transformer.encode(documents)

    return embeddings

In [ ]:
def retrieve_documents(query: str):
  # Use the stored documents to retrieve the most similar documents to the query

  # Retrieval of Ducuments
  with open("documents.txt", 'r') as fh:
    documents = fh.readlines()
    fh.close()

  # Loading the embeddings
  embeddings = embed_documents(documents)

  # Create the embedding for the query
  embed_query = embed_documents([query])

  # Compute similarity between query and embeddings
  similarities = sentence_transformer.similarity(embed_query, embeddings)

  # Indices of most similar documents
  idx = torch.argmax(similarities)

  return documents[idx]

In [ ]:
def generate_response(query: str, documents: list[str]):
  # Use the LLM to generate a response

  prompt = f"""
    Context information is below:
    -------------------------------
    {''.join(documents)}
    -------------------------------
    Given the context information above and not prior knowledge, answer the query of the human.
    ### Human:{query}
    ### Assistant:"""


  # Loading the tokenizer of mistral
  tokenizer = AutoTokenizer.from_pretrained("mistralai/Mistral-7B-v0.3")

  # Loading mistral
  llm = AutoModelForCausalLM.from_pretrained(
      "mistralai/Mistral-7B-v0.3",
      torch_dtype=torch.float16
  )

  # Loading the fine-tuned adapter
  llm.load_adapter("genaibook/fine_tune_e2e")

  # Creation of the pipeline
  pipe = pipeline(
      "text-generation",
      model=llm,
      tokenizer=tokenizer
  )

  # Generation of teh answer
  return pipe(prompt, max_new_tokens=100)

In [ ]:
def pipeline_gen(query: str):
  documents = retrieve_documents(query)
  response = generate_response(query, documents)
  return response

## Embedding documents

Creation of some documents about news in 2025:

In [ ]:
documents = ['The highest-grossing movie of 2025 so far is "Ne Zha 2", a Chinese animated film that has grossed over $948 million worldwide\n',
 '"A Minecraft Movie" is the second highest-grossing film, with a worldwide gross of $423,822,028\n',
 '"Lilo & Stitch" is the third highest-grossing movie with a $632 million gross\n',
 'Paris Saint-Germain won UEFA Champions League in 2025\n',
 'The president of USA in 2025 was Donald Trump\n',
 'Israel launched blistering attacks on the heart of Iran’s nuclear and military structure in 2025'
]

In [ ]:
## TO WRITE THE DOCUMENTS TO RETRIEVE IN A FILE
with open("documents.txt", 'a') as fh:
  fh.writelines(documents)
  fh.close()

## RAG

In [ ]:
query = "What is the movie with the biggest gross in 2025?"
pipeline_gen(query)

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

Batches:   0%|          | 0/1 [00:00<?, ?it/s]

tokenizer_config.json:   0%|          | 0.00/137k [00:00<?, ?B/s]

tokenizer.model:   0%|          | 0.00/587k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.96M [00:00<?, ?B/s]

...

generation_config.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

adapter_config.json:   0%|          | 0.00/650 [00:00<?, ?B/s]

adapter_model.safetensors:   0%|          | 0.00/13.6M [00:00<?, ?B/s]

Device set to use cuda:0
Setting `pad_token_id` to `eos_token_id`:2 for open-end generation.


[{'generated_text': '\n    Context information is below:\n    -------------------------------\n    The highest-grossing movie of 2025 so far is "Ne Zha 2", a Chinese animated film that has grossed over $948 million worldwide\n\n    -------------------------------\n    Given the context information above and not prior knowledge, answer the query of the human.\n    ### Human:What is the movie with the biggest gross in 2025?\n    ### Assistant:The movie with the biggest gross in 2025 is "Ne Zha 2", a Chinese animated film that has grossed over $948 million worldwide.'}]